# NumPilot 全新机器源码验收

本 Notebook 不使用 wheelhouse。只克隆 NumPilot，AsNumpy 从固定官方源码现场编译，CANNBot 从固定提交拉取并校验。NumPilot 锁定 profile 与独立审计的官方最新提交中，公开 `matmul` 都是 CPU fallback；本轮若接受它，只验证兼容性，另行授权后才做数值实跑，不发布 NPU 加速比。

In [ ]:
from pathlib import Path
import subprocess

REPO = Path.cwd().resolve()
assert (REPO / 'install.sh').is_file(), '请先进入刚克隆的 NumPilot 仓库'
print(REPO)


## 显式同意联网与 CANNBot 许可证

先审阅固定 commit `1ef6372a538f39b0b7ed34666a3e1f465fb4936a` 的 CANN Open Software License v2.0。只有确实接受并允许本次联网时，才把下面两个值改成 `True`。若默认私有 Python 镜像缺包，由用户显式向安装命令附加 `--pip-index-url https://pypi.org/simple`；安装器不会悄悄换源。

In [ ]:
ALLOW_NETWORK = False
ACCEPT_CANNBOT_LICENSE = False
assert ALLOW_NETWORK, '未允许安装器联网，停止'
assert ACCEPT_CANNBOT_LICENSE, '未接受 CANNBot 许可证，停止'
subprocess.run([
    str(REPO / 'install.sh'),
    '--allow-network',
    '--with-cannbot',
    '--accept-cannbot-license',
], cwd=REPO, check=True)


## 离线复验

下列命令使用仓库 `.venv`，验证 AsNumpy README smoke、CANNBot Skill 清单与 NPU 环境。

In [ ]:
PYTHON = REPO / '.venv' / 'bin' / 'python'
subprocess.run([str(PYTHON), '-m', 'numpilot', '--version'], check=True)
subprocess.run([
    str(PYTHON), '-m', 'numpilot', 'bootstrap', '--verify',
    '--with-cannbot', '--accept-cannbot-license',
], check=True)
subprocess.run([str(PYTHON), '-m', 'numpilot', 'doctor', '--backend', 'npu'], check=True)


## GLM-5.3 与迁移停点

克隆不会生成或复制 `llm.json`。在终端运行 `python -m numpilot configure-llm --init`，填写新机器自己的 endpoint、Token、`model=glm-5.3`、`provider=glm`，再执行 `--check` 和 `--probe`。代理只在需要时从当前进程注入，不写入仓库。

上传待测纯 NumPy 文件后先报告安装与能力分类；在用户明确说“开始迁移”前不要调用 GLM。Case01 必须由用户显式接受 `matmul` fallback，结果固定 `strict_npu=false`、`performance_valid=false`，且第一轮不得继续 Case02–10。